In [49]:
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from deltalake import DeltaTable, write_deltalake
import polars as pl

In [25]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000", # MinIO API endpoint
    aws_access_key_id="adminminio", # User name
    aws_secret_access_key="adminminio", # Password
)

In [27]:
buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]

def createBucket(name, list_buckets):
    if name in list_buckets:
        print(f"Bucket '{name}' already exists!")
    else:
        s3.create_bucket(Bucket=name)
        print(f"Created bucket: {name}")
createBucket("deltalake", buckets)

Created bucket: deltalake


In [30]:
# Create two sub-buckets inside landing_zone.
s3.put_object(Bucket="deltalake", Key="bronze/") # bronze layer


{'ResponseMetadata': {'RequestId': '189D125B24E7C307',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': 'AAAAAA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '189D125B24E7C307',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '4398',
   'x-ratelimit-remaining': '4398',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 15 Mar 2026 17:00:10 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
 'ChecksumCRC32': 'AAAAAA==',
 'ChecksumType': 'FULL_OBJECT'}

In [12]:
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT 'minio:9000',
    KEY_ID 'adminminio',
    SECRET 'adminminio',
    URL_STYLE 'path',
    USE_SSL false
);""")

In [52]:
datasets = {
    "co2-emission": "s3://landing-zone/persistent-landing/csv/co2-emission*.csv",
    "global_warming": "s3://landing-zone/persistent-landing/csv/global_warming*.csv",
    "natural_disaster_tweets": "s3://landing-zone/persistent-landing/csv/natural_disaster_tweets*.csv",
    "temperature_change": "s3://landing-zone/persistent-landing/csv/temperature_change*.csv"
}
BRONZE_BASE_PATH = "s3://deltalake/bronze"
MINIO_HOST = "minio:9000"
MINIO_ACCESS_KEY = "adminminio"
MINIO_SECRET_KEY = "adminminio"

def ingest_with_duckdb():
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    con.execute(f"""
        SET s3_endpoint='{MINIO_HOST}';
        SET s3_access_key_id='{MINIO_ACCESS_KEY}';
        SET s3_secret_access_key='{MINIO_SECRET_KEY}';
        SET s3_use_ssl=false;
        SET s3_url_style='path';
    """)




    for table_name, s3_path in datasets.items():
        print(f" DuckDB processing: {table_name}...")
        try:
            output_path = f"{BRONZE_BASE_PATH}/{table_name}.parquet"
            
            con.execute(f"""
                COPY (
                    SELECT * FROM read_csv_auto('{s3_path}')
                ) TO '{output_path}' (FORMAT PARQUET);
            """)
            
            print(f"✅ Saved to: {output_path}")
            
        except Exception as e:
            print(f"❌ DuckDB failed for {table_name}: {e}")
ingest_with_duckdb()

 DuckDB processing: co2-emission...
✅ Saved to: s3://deltalake/bronze/co2-emission.parquet
 DuckDB processing: global_warming...
✅ Saved to: s3://deltalake/bronze/global_warming.parquet
 DuckDB processing: natural_disaster_tweets...
✅ Saved to: s3://deltalake/bronze/natural_disaster_tweets.parquet
 DuckDB processing: temperature_change...
✅ Saved to: s3://deltalake/bronze/temperature_change.parquet


In [55]:

con.execute(f"""
    SET s3_endpoint='{MINIO_HOST}';
    SET s3_access_key_id='{MINIO_ACCESS_KEY}';
    SET s3_secret_access_key='{MINIO_SECRET_KEY}';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

print("🔎 Reading first 10 rows via DuckDB:")
table_path = "s3://deltalake/bronze/co2-emission.parquet"
df_view = con.execute(f"SELECT * FROM read_parquet('{table_path}') LIMIT 10").df()

print(df_view)

🔎 Reading first 10 rows via DuckDB:
    Make       Model Vehicle Class  Engine Size(L)  Cylinders Transmission  \
0  ACURA         ILX       COMPACT             2.0          4          AS5   
1  ACURA         ILX       COMPACT             2.4          4           M6   
2  ACURA  ILX HYBRID       COMPACT             1.5          4          AV7   
3  ACURA     MDX 4WD   SUV - SMALL             3.5          6          AS6   
4  ACURA     RDX AWD   SUV - SMALL             3.5          6          AS6   
5  ACURA         RLX      MID-SIZE             3.5          6          AS6   
6  ACURA          TL      MID-SIZE             3.5          6          AS6   
7  ACURA      TL AWD      MID-SIZE             3.7          6          AS6   
8  ACURA      TL AWD      MID-SIZE             3.7          6           M6   
9  ACURA         TSX       COMPACT             2.4          4          AS5   

  Fuel Type  Fuel Consumption City (L/100 km)  \
0         Z                               9.9   
1      

In [59]:
# Connection configuration for MinIO/S3
# Using the standard storage_options compatible with delta-rs
storage_options = {
    "AWS_ACCESS_KEY_ID": "adminminio",
    "AWS_SECRET_ACCESS_KEY": "adminminio",
    "AWS_ENDPOINT_URL": "http://minio:9000",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

# List of Parquet files currently sitting in the bronze bucket root
parquet_files = [
    "co2-emission.parquet",
    "global_warming.parquet",
    "natural_disaster_tweets.parquet",
    "temperature_change.parquet"
]

def finalize_bronze_layer():
    """
    Reads existing Parquet files and persists them as Delta Tables.
    This process creates the required folder structure and the _delta_log (Version 0).
    """
    for file_name in parquet_files:
        # Generate a clean table name (replacing hyphens with underscores)
        table_name = file_name.replace(".parquet", "").replace("-", "_")
        
        # Define source path (raw parquet) and target path (delta folder)
        source_path = f"s3://deltalake/bronze/{file_name}"
        target_folder = f"s3://deltalake/bronze/{table_name}/"
        
        print(f"📦 Finalizing Bronze Table: {table_name}...")
        
        try:
            # Load the parquet file into memory
            df = pl.read_parquet(source_path, storage_options=storage_options)
            
            # Write as Delta Table to create the versioned directory structure
            # Mode 'overwrite' ensures a fresh Version 0 is created
            write_deltalake(
                target_folder,
                df,
                mode="overwrite",
                storage_options=storage_options
            )
            print(f"✅ Successfully created Delta folder and log at: {target_folder}")
            s3.delete_object(
                Bucket='deltalake', 
                Key=f"bronze/{file_name}"
            )
        except Exception as e:
            print(f"❌ Failed to finalize {table_name}: {e}")

finalize_bronze_layer()

📦 Finalizing Bronze Table: co2_emission...
✅ Successfully created Delta folder and log at: s3://deltalake/bronze/co2_emission/
📦 Finalizing Bronze Table: global_warming...
✅ Successfully created Delta folder and log at: s3://deltalake/bronze/global_warming/
📦 Finalizing Bronze Table: natural_disaster_tweets...
✅ Successfully created Delta folder and log at: s3://deltalake/bronze/natural_disaster_tweets/
📦 Finalizing Bronze Table: temperature_change...
✅ Successfully created Delta folder and log at: s3://deltalake/bronze/temperature_change/
